# Model & DB loading

In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from FlagEmbedding import BGEM3FlagModel

# load the model and processor
ckpt_siglip2 = "google/siglip2-so400m-patch14-384"
ckpt_bgem3 = "BAAI/bge-m3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor_siglip2 = AutoProcessor.from_pretrained(ckpt_siglip2)
model_siglip2 = AutoModel.from_pretrained(
    ckpt_siglip2,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" # Install flash-attn before using this
).eval()
model_bgem3 = BGEM3FlagModel(
    ckpt_bgem3,
    normalize_embeddings=True,
    use_bf16=True,
    devices=device,
    return_dense=True,
    return_sparse=True,
    model_kwargs={"attn_implementation": "flash_attention_2"}
)

C:\Users\Bui Thien Nghia\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url='http://localhost:6333')

# Query encoding

In [4]:
QUERY = "Đây là cảnh một người phụ nữ mặc áo hoa và đội nón lá đang cúi người đậy nắp cho một chiếc lu gốm lớn. Xung quanh bà là rất nhiều chiếc lu gốm và vại sành kích thước khác nhau được xếp san sát ngoài trời, một số chiếc đã được đậy kín nắp trong khi số khác vẫn đang mở."
inputs = processor_siglip2(text=QUERY, padding='max_length', truncation=True, return_tensors='pt')

with torch.no_grad():
    query_siglip2_dense = model_siglip2.get_text_features(**inputs.to(device)).pooler_output
    query_siglip2_dense = F.normalize(query_siglip2_dense, dim=-1)

query_siglip2_dense = query_siglip2_dense.to('cpu').squeeze().tolist()

query_bgem3 = model_bgem3.encode(QUERY)
query_bgem3_dense = query_bgem3['dense_vecs'].squeeze().tolist()
query_bgem3_sparse = models.SparseVector(
    indices=np.array(list(query_bgem3['lexical_weights'].keys()), dtype=np.int32).tolist(),
    values=np.array(list(query_bgem3['lexical_weights'].values()), dtype=np.float32).tolist(),
)

query_dict = {
    "siglip2_dense": query_siglip2_dense,
    "bgem3_dense": query_bgem3_dense,
    "bgem3_sparse": query_bgem3_sparse
}

In [ ]:
COLLECTION_NAME = 'test'
LIMIT = 50

search_params = models.SearchParams(
    hnsw_ef=200,
    quantization=models.QuantizationSearchParams(
        rescore=True,
        oversampling=2.0
    )
)

prefetch = [
    models.Prefetch(
        query=query_siglip2_dense,
        using="dense_siglip2",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_bgem3_dense,
        using="dense_bgem3",
        limit=LIMIT * 2
    ),
    models.Prefetch(
        query=query_bgem3_sparse,
        using='sparse_bgem3',
        limit=LIMIT * 2
    )
]

def calculate_neighbor_aggregated_score(point: models.ScoredPoint, query_vectors: dict, neighbor_limit: int, aggregation_weight: float = 0.5):
    # Summarize candidate's info
    candidate_info = {
        "original_score": point.score,
        "vid_id": point.payload["vid_id"],
        "time_order": float(point.payload["time_order"]),
        "siglip2_dense": point.vector["dense_siglip2"],
    }

    # Get valid ANN
    neighbors = client.query_points(
        collection_name=COLLECTION_NAME,
        query=candidate_info["siglip2_dense"],
        using="dense_siglip2",
        limit=limit + 1, # Exclude itself out later
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="vid_id",
                    match=models.MatchValue(value=candidate_info["vid_id"])
                )
            ]
        ),
        with_payload=["time_order"]
    ).points

    # Get optimal delta
    neighbor_time_orders = [float(neighbor.payload["time_order"]) for neighbor in neighbors]
    delta = (max(neighbor_time_orders) - min(neighbor_time_orders)) / (len(neighbors) - 1)

    # Calculate visual and temporal relevance score
    total_temporal_relevance_score = 0
    total_visual_relevance_score = 1
    for neighbor in neighbors:
        neighbor_prefetch = [
            models.Prefetch(
                query=query_vectors["siglip2_dense"],
                using="dense_siglip2",
                limit=1
            ),
            models.Prefetch(
                query=query_vectors["bgem3_dense"],
                using="dense_bgem3",
                limit=1
            ),
            models.Prefetch(
                query=query_vectors["bgem3_sparse"],
                using='sparse_bgem3',
                limit=1
            )
        ]

        # Visual relevance score
        neighbor_visual_relevance_score = neighbor.score
        total_visual_relevance_score += neighbor_visual_relevance_score

        # Temporal relevance score
        neighbor_temporal_relevance_score = client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=neighbor_prefetch,
            query=models.FormulaQuery(
                formula=models.SumExpression(sum=[
                    models.MultExpression(mult=[0.6, "$score[0]"]),
                    models.MultExpression(mult=[0.2, "$score[0]"]),
                    models.MultExpression(mult=[0.2, "$score[0]"]),
                ])
            ),
            query_filter=models.Filter(
                must=[models.HasIdCondition(has_id=[neighbor.id])]
            ),
            limit=1
        ).points[0].score
        
        neighbor_temporal_relevance_score *= np.exp(
            - np.square(candidate_info["time_order"] - float(neighbor.payload["time_order"]))
            / (2 * np.square(delta))
        ).astype(float)
        total_temporal_relevance_score += neighbor_temporal_relevance_score

    # Calculate aggregated score
    aggregation_component = (np.log(total_visual_relevance_score) / (len(neighbors) - 1)) * total_temporal_relevance_score
    return (1 - aggregation_weight) * candidate_info["original_score"] + aggregation_weight * aggregation_component

from IPython.display import display
import ipywidgets as widgets

points = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=prefetch,
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    with_vectors=["dense_siglip2"],
    with_payload=True,
    limit=LIMIT,
    search_params=search_params
).points

aggregated_points = [[point, calculate_neighbor_aggregated_score(point, query_dict, 10)] for point in points]
aggregated_points = [pair[0] for pair in sorted(aggregated_points, key=lambda x : x[1], reverse=True)]

relevant_img_paths = [widgets.Image(value=open(point.payload['img_path'], 'rb').read(), format='png', width=315) for point in aggregated_points]
cols = 5
for i in range(0, len(relevant_img_paths), cols):
    if i + cols > len(relevant_img_paths):
        images = relevant_img_paths[i:len(relevant_img_paths)]
    else:
        images = relevant_img_paths[i:i+cols]

    row = widgets.HBox(images)
    display(row)
    # print([hit.payload['img_path'].split('\\', 5)[-1].replace('\\', '_').replace(".jpg", '') for hit in hits[i:i+cols]])

# Retrieval test

In [13]:
dict(hits[0])['vector']

{'sparse_bgem3': SparseVector(indices=[4, 5, 7, 106, 116, 284, 544, 681, 1000, 3531, 4225, 7099, 7756, 8086, 8192, 9611, 10657, 10895, 11161, 13407, 14868, 15171, 17749, 21243, 30692, 32228, 39469, 43561, 46175, 150535, 191471], values=[0.022094727, 0.052490234, 0.07128906, 0.0703125, 0.21679688, 0.10058594, 0.05126953, 0.011291504, 0.014343262, 0.10546875, 0.083984375, 0.08544922, 0.18554688, 0.030517578, 0.0859375, 0.08105469, 0.07861328, 0.012817383, 0.09472656, 0.083984375, 0.16503906, 0.13769531, 0.057128906, 0.12597656, 0.106933594, 0.0859375, 0.13378906, 0.110839844, 0.10986328, 0.17578125, 0.25390625]),
 'dense_siglip2': [0.022189375,
  -0.0030937109,
  0.00051053846,
  -0.023286652,
  -0.0024079129,
  -0.03364982,
  0.045841783,
  -0.038526606,
  0.04754866,
  0.009692611,
  -0.021579776,
  -0.021092098,
  0.12240732,
  -0.051937766,
  -0.015849553,
  0.032430623,
  0.010424129,
  -0.04047732,
  -0.0013030161,
  0.046573304,
  0.06437357,
  -0.25164214,
  0.042671874,
  -0.011